# Homework 2: Vector Search

We embed lesson pages with a lightweight ONNX model, then search them by hand, with `minsearch`, and with hybrid search (RRF).

## Setup

We use the `Embedder` class from [embed/embedder.py](embed/embedder.py), which wraps an ONNX Runtime session + tokenizer. The model was already downloaded with `embed/download.py` into `embed/models/Xenova/all-MiniLM-L6-v2`.

In [1]:
from embed.embedder import Embedder

embedder = Embedder(path="embed/models/Xenova/all-MiniLM-L6-v2")

2026-06-23 18:56:33.394116543 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


## Q1. Embedding a query

We embed the query and look at `v[0]`, the first component of the 384-dim vector.

In [2]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(v.shape)
print(f"Q1 answer (v[0]): {v[0]:.4f}")

(384,)
Q1 answer (v[0]): -0.0206


Closest option: **-0.02**.

## Loading the data

We pull the 72 lesson pages from the course repo, pinned to commit `8c1834d`, same as in homework 1.

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


## Q2. Cosine similarity

The embedder normalizes its output vectors, so the dot product between two embeddings *is* their cosine similarity. We embed the `07-sqlitesearch-vector.md` page and compare it with the Q1 query vector.

In [4]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(d for d in documents if d["filename"] == target_filename)

doc_vector = embedder.encode(target_doc["content"])
similarity = v.dot(doc_vector)

print(f"Q2 answer (cosine similarity): {similarity:.4f}")

Q2 answer (cosine similarity): 0.3611


Closest option: pick whichever of `0.07 / 0.37 / 0.68 / 0.92` is nearest to the printed value.

## Q3. Chunking and search by hand

A full page mixes several topics, which dilutes its embedding. We chunk every page into overlapping windows (`size=2000`, `step=1000`), embed each chunk, stack the vectors into a matrix `X`, and score the Q1 query against every chunk with a single dot product: `scores = X.dot(v)`.

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [6]:
chunk_texts = [c["content"] for c in chunks]
X = embedder.encode_batch(chunk_texts)
print(X.shape)

(295, 384)


In [7]:
scores = X.dot(v)
best_idx = scores.argmax()
best_chunk = chunks[best_idx]

print(f"Q3 answer (top chunk's filename): {best_chunk['filename']}")
print(f"score: {scores[best_idx]:.4f}")

Q3 answer (top chunk's filename): 02-vector-search/lessons/07-sqlitesearch-vector.md
score: 0.6489


## Q4. Vector search with minsearch

Doing vector search by hand is good for learning, but in practice we reach for a library. We index the chunk embeddings with `minsearch.VectorSearch` and run a query through it.

In [8]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

q4_results = vindex.search(q4_vector, num_results=5)

for r in q4_results:
    print(r["filename"])

print()
print(f"Q4 answer (first result): {q4_results[0]['filename']}")

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md

Q4 answer (first result): 04-evaluation/lessons/05-search-metrics.md


## Q5. Text search vs vector search

Vector search matches by meaning; keyword search matches exact words. We build a text index over the same chunks (with `content` as the text field) and compare the top-5 results from each method for the same query.

In [9]:
from minsearch import Index

tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)

vector_results = vindex.search(q5_vector, num_results=5)
text_results = tindex.search(q5_query, num_results=5)

vector_filenames = [r["filename"] for r in vector_results]
text_filenames = [r["filename"] for r in text_results]

print("Vector results:")
for f in vector_filenames:
    print(" ", f)

print("Text results:")
for f in text_filenames:
    print(" ", f)

only_in_vector = [f for f in vector_filenames if f not in text_filenames]
print()
print(f"Q5 answer (in vector but not text): {only_in_vector}")

Vector results:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
Text results:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

Q5 answer (in vector but not text): ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


## Q6. Hybrid search

Vector search misses exact terms; text search misses paraphrases. Hybrid search merges both ranked lists with Reciprocal Rank Fusion (RRF): every document gets `1 / (k + rank)` from each list it appears in, summed across lists, with `k = 60`.

In [10]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [11]:
q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)

q6_vector_results = vindex.search(q6_vector, num_results=5)
q6_text_results = tindex.search(q6_query, num_results=5)

hybrid_results = rrf([q6_vector_results, q6_text_results])

for r in hybrid_results:
    print(r["filename"])

print()
print(f"Q6 answer (first after RRF): {hybrid_results[0]['filename']}")

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md

Q6 answer (first after RRF): 01-agentic-rag/lessons/13-function-calling.md


Notice that the top hybrid result needn't be first in either individual list — RRF rewards documents that rank reasonably well in *both* lists.